# RNN Regularization: Dropout for LSTMs

This notebook explores regularization techniques for RNNs, centered on the paper
**"Recurrent Neural Network Regularization"** (Zaremba et al., 2014).

The paper's core contribution: apply dropout **only to non-recurrent connections**
(between layers), never to recurrent connections (across time). We also implement
layer normalization, weight decay, and early stopping — standard techniques that
complement dropout in practice (these go beyond the paper).

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from implementation import (
    dropout_forward, dropout_backward,
    layer_norm_forward, layer_norm_backward,
    RegularizedLSTM, EarlyStoppingMonitor,
    compute_l2_penalty, regularized_loss,
    LSTMLanguageModel, clip_grad_norm,     # the paper's model (Part 6)
)
from visualization import (
    plot_learning_curves,
    plot_regularization_comparison,
    plot_weight_distributions
)
from train_minimal import load_ptb, batchify, train_lm

np.random.seed(42)
print("All modules loaded.")

## Part 1: Understanding Overfitting

The core problem this paper addresses: without regularization, larger LSTMs overfit
badly. The non-regularized model achieves 114.5 perplexity on PTB, but making it
bigger doesn't help — it just memorizes training data. With proper dropout, the
paper shows bigger models keep improving (down to 78.4 perplexity).

In [ ]:
# Simulate overfitting scenario
epochs = range(30)

# Without regularization: training loss decreases, validation loss increases
train_loss_overfit = 2.0 / (1 + np.array(epochs) * 0.1) + np.random.normal(0, 0.05, len(epochs))
val_loss_overfit = 2.0 / (1 + np.array(epochs) * 0.05) + np.random.normal(0, 0.08, len(epochs))
val_loss_overfit[15:] = val_loss_overfit[15:] + np.array(epochs[15:]) * 0.02

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss_overfit, 'b-o', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss_overfit, 'r-s', label='Validation Loss', linewidth=2)
plt.axvline(x=15, color='green', linestyle='--', alpha=0.7, label='Overfitting starts')
plt.fill_between(epochs[15:], train_loss_overfit[15:], val_loss_overfit[15:], alpha=0.2, color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('WITHOUT Regularization\n(Overfitting Gap Grows)')
plt.legend()
plt.grid(True, alpha=0.3)

# With regularization: both decrease together
train_loss_regular = 2.0 / (1 + np.array(epochs) * 0.08) + np.random.normal(0, 0.04, len(epochs))
val_loss_regular = 2.0 / (1 + np.array(epochs) * 0.07) + np.random.normal(0, 0.05, len(epochs))

plt.subplot(1, 2, 2)
plt.plot(epochs, train_loss_regular, 'b-o', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss_regular, 'r-s', label='Validation Loss', linewidth=2)
plt.fill_between(epochs, train_loss_regular, val_loss_regular, alpha=0.2, color='green')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('WITH Regularization\n(Controlled, Stable)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: Without regularization - BIG GAP = overfitting")
print("Right: With regularization - SMALL GAP = good generalization")

## Part 2: Dropout — The Paper's Core Contribution

The key insight from Zaremba et al.: apply dropout only to **non-recurrent**
connections (between layers), not to recurrent connections (across timesteps).
This prevents corruption of the LSTM's temporal memory while still getting
the regularization benefits of dropout.

In [ ]:
# Create some activations
activations = np.random.randn(10)

print("Original activations:")
print(activations.round(2))

# Apply dropout during training
output_train, mask = dropout_forward(activations, keep_prob=0.8, training=True)
print(f"\nWith dropout (keep_prob=0.8):")
print(output_train.round(2))
print(f"Mask: {mask.round(2)}")
print(f"Notice: Some values are 0, others are scaled up")

# No dropout during testing
output_test, _ = dropout_forward(activations, keep_prob=0.8, training=False)
print(f"\nDuring testing (training=False):")
print(output_test.round(2))
print(f"All values pass through unchanged!")

In [ ]:
# Visualize dropout effect
activations_matrix = np.random.randn(20, 50)

plt.figure(figsize=(14, 5))

# Without dropout
plt.subplot(1, 2, 1)
plt.imshow(activations_matrix, cmap='RdYlGn', aspect='auto')
plt.colorbar(label='Activation value')
plt.title('WITHOUT Dropout\n(All neurons active)', fontsize=12, fontweight='bold')
plt.xlabel('Hidden units')
plt.ylabel('Time steps')

# With dropout
dropped_matrix = np.zeros_like(activations_matrix)
for i in range(activations_matrix.shape[0]):
    dropped_matrix[i], _ = dropout_forward(activations_matrix[i], keep_prob=0.7, training=True)

plt.subplot(1, 2, 2)
plt.imshow(dropped_matrix, cmap='RdYlGn', aspect='auto')
plt.colorbar(label='Activation value')
plt.title('WITH Dropout (keep_prob=0.7)\n(30% randomly deactivated)', fontsize=12, fontweight='bold')
plt.xlabel('Hidden units')
plt.ylabel('Time steps')

plt.tight_layout()
plt.show()

print(f"Black areas = deactivated neurons")
print(f"Dropout forces network to learn from different subsets of neurons")
print(f"This prevents co-adaptation and improves generalization")

## Part 3: Layer Normalization (Our Addition — Ba et al. 2016)

Layer normalization is NOT from the Zaremba paper — it was published 2 years later.
We include it because it's commonly paired with dropout in modern RNN implementations.
It normalizes activations per-sample to zero mean and unit variance.

In [ ]:
# Generate activations from untrained network
batch_size = 32
hidden_size = 64

# Random activations (before training)
activations_raw = np.random.randn(batch_size, hidden_size) * 10  # Large variance

print("Raw activations statistics:")
print(f"  Mean: {activations_raw.mean(axis=1).mean():.4f}")
print(f"  Std: {activations_raw.std(axis=1).mean():.4f}")
print(f"  Min: {activations_raw.min():.4f}")
print(f"  Max: {activations_raw.max():.4f}")

# Apply layer normalization
gamma = np.ones((hidden_size,))
beta = np.zeros((hidden_size,))

activations_norm, cache = layer_norm_forward(activations_raw, gamma, beta)

print("\nAfter layer normalization:")
print(f"  Mean: {activations_norm.mean(axis=1).mean():.4f}")
print(f"  Std: {activations_norm.std(axis=1).mean():.4f}")
print(f"  Min: {activations_norm.min():.4f}")
print(f"  Max: {activations_norm.max():.4f}")
print("\nMuch more stable: Mean ~ 0, Std ~ 1")

In [ ]:
# Visualize layer norm effect
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(activations_raw.flatten(), bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Raw Activations\n(Unstable: large range)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Activation value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[0].legend()

axes[1].hist(activations_norm.flatten(), bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].set_title('After Layer Norm\n(Stable: centered at 0, std=1)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Activation value')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero')
axes[1].legend()
axes[1].set_xlim([-4, 4])

plt.tight_layout()
plt.show()

print("Layer Normalization benefits:")
print("  1. Prevents gradient explosion (gradients stay in reasonable range)")
print("  2. Speeds up convergence (stable gradient flow)")
print("  3. Allows higher learning rates")
print("  4. Reduces dependence on weight initialization")

## Part 4: Weight Decay / L2 Regularization (Our Addition)

Weight decay is not discussed in the Zaremba paper, but it's a standard
regularization technique: add a penalty proportional to squared weight magnitudes.
This encourages smaller weights and simpler models.

In [ ]:
# Simulate weight evolution
np.random.seed(42)
initial_weights = np.random.randn(1000) * 0.01

# Training without weight decay: weights grow large
weights_no_decay = initial_weights.copy()
for epoch in range(50):
    weights_no_decay += np.random.randn(1000) * 0.05  # Gradient updates

# Training with weight decay: weights stay small
weights_with_decay = initial_weights.copy()
weight_decay_coeff = 0.01
for epoch in range(50):
    weights_with_decay += np.random.randn(1000) * 0.05  # Gradient updates
    weights_with_decay -= weight_decay_coeff * weights_with_decay  # L2 penalty pulls toward zero

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(initial_weights, bins=50, alpha=0.7, color='blue', edgecolor='black')
axes[0].set_title('Initial Weights\n(Small random)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Weight value')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)

axes[1].hist(weights_no_decay, bins=50, alpha=0.7, color='orange', edgecolor='black')
axes[1].set_title('After Training\n(NO Weight Decay - Large spread)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Weight value')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2)

axes[2].hist(weights_with_decay, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[2].set_title('After Training\n(WITH Weight Decay - Concentrated near 0)', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Weight value')
axes[2].set_ylabel('Frequency')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

print("Weight Decay Statistics:")
print(f"Without decay: mean={weights_no_decay.mean():.4f}, std={weights_no_decay.std():.4f}")
print(f"With decay:    mean={weights_with_decay.mean():.4f}, std={weights_with_decay.std():.4f}")
print("\nWeight decay encourages simple models (Occam's Razor)")

In [ ]:
# Show weight decay formula
print("Weight Decay (L2 Regularization) Formula:")
print()
print("  Loss_total = Loss_model + λ * (1/2) * Σ(w²)")
print()
print("Where:")
print(f"  Loss_model = cross-entropy or other model loss")
print(f"  λ (lambda) = weight decay coefficient (e.g., 0.0001)")
print(f"  Σ(w²) = sum of squared weights")
print()
print("Typical values:")
print(f"  Light:   λ = 0.00001 (very gentle)")
print(f"  Normal:  λ = 0.0001  (standard)")
print(f"  Strong:  λ = 0.001   (heavy regularization)")

# Example
print("\nExample:")
model_loss = 0.5
weights_example = [np.random.randn(10), np.random.randn(20)]
weight_decay = 0.0001

l2_penalty = compute_l2_penalty(weights_example, weight_decay)
total_loss = regularized_loss(model_loss, weights_example, weight_decay)

print(f"  Model loss: {model_loss:.4f}")
print(f"  L2 penalty: {l2_penalty:.6f}")
print(f"  Total loss: {total_loss:.4f}")

## Part 5: Early Stopping (Our Addition)

Early stopping monitors validation loss and stops training when it stops improving.
Not formally discussed in the Zaremba paper, but they clearly used fixed training
schedules (39 epochs for medium, 55 for large). Early stopping automates the
decision of when to stop.

In [ ]:
# Simulate training with early stopping
epochs_data = [
    (0, 2.50, 2.51),
    (1, 2.00, 2.10),
    (2, 1.50, 1.80),
    (3, 1.00, 1.70),
    (4, 0.70, 1.75),  # Validation stops improving
    (5, 0.50, 1.85),
    (6, 0.35, 2.00),
    (7, 0.25, 2.15),  # Would stop here with patience=3
]

epochs, train_losses, val_losses = zip(*epochs_data)

monitor = EarlyStoppingMonitor(patience=3, verbose=False)

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, 'b-o', linewidth=2.5, markersize=8, label='Training Loss')
plt.plot(epochs, val_losses, 'r-s', linewidth=2.5, markersize=8, label='Validation Loss')

# Mark best epoch
best_epoch = np.argmin(val_losses)
plt.scatter([best_epoch], [val_losses[best_epoch]], color='green', s=300, marker='*', 
           zorder=5, label=f'Best (epoch {best_epoch})')

# Mark early stopping
stop_epoch = 7
plt.axvline(x=stop_epoch, color='orange', linestyle='--', linewidth=2.5, 
            label=f'Early Stop (patience=3, epoch {stop_epoch})')

# Shade regions
plt.axvspan(-0.5, best_epoch + 0.5, alpha=0.1, color='green', label='Improving')
plt.axvspan(best_epoch + 0.5, stop_epoch + 0.5, alpha=0.1, color='red', label='No improvement')

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Early Stopping: Stop Before Overfitting', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Early Stopping Rules:")
print(f"1. Best validation loss: epoch {best_epoch} (loss={val_losses[best_epoch]:.2f})")
print(f"2. Validation stopped improving after epoch {best_epoch}")
print(f"3. With patience=3, we wait 3 epochs with no improvement")
print(f"4. Stop at epoch {stop_epoch} and load best model state")

## Part 6: Training the Paper's Model on Penn Treebank

Parts 1-5 used simulated curves and toy inputs. Here we train the model the paper actually trains
(Sec. 4.1): word embeddings → 2 LSTM layers → softmax over the 10,000-word Penn Treebank vocabulary,
with dropout on every non-recurrent connection (embedding → layer 1, layer 1 → layer 2,
layer 2 → softmax) and never on the recurrent $h_{t-1} \to h_t$. `LSTMLanguageModel` in
`implementation.py` has a hand-written backward pass (truncated backpropagation through time) that
matches `torch.autograd` to about 1e-15 (`validation/day03_bptt_check.py`).

The metric is the paper's: **perplexity** = exp(average cross-entropy per word). A uniform guess over
the vocabulary scores 10,000; the paper's best single model scores 78.4 on the test set.

**Budget.** The paper's models (2×650 and 2×1500 units, 39-55 epochs over 929k words) took half a day
to a day on a GPU. This part has well under a minute of CPU time, so it trains the same 2×200 model as
`python train_minimal.py` for 3 epochs on the first 50,000 training words and validates on the first
20,000 validation words. Layer norm, weight decay and early stopping (Parts 3-5) are not used: the
paper uses none of them.

In [ ]:
# Penn Treebank, Mikolov's split from the paper's repository (github.com/wojzaremba/lstm).
# load_ptb() reads data/ptb/ (or $PTB_DIR) and downloads the three files (~6 MB) the first time.
# Like the paper's data.lua, every line end becomes an <eos> token.
train_ids, valid_ids, test_ids, vocab = load_ptb()
id2word = {i: w for w, i in vocab.items()}          # invert the dict: id -> word
print(f"train {len(train_ids):,} / valid {len(valid_ids):,} / test {len(test_ids):,} words; "
      f"vocabulary {len(vocab):,}")
print("first 25 training words:", " ".join(id2word[i] for i in train_ids[:25]))

In [ ]:
# One training step by hand: forward over a 35-step window, BPTT backward, clip, SGD update.
lm = LSTMLanguageModel(len(vocab), hidden_size=200, num_layers=2, keep_prob=0.5, init_scale=0.1, seed=0)
data = batchify(train_ids[:50_000], 20)     # (2500, 20): 20 contiguous word streams side by side
x, y = data[:35], data[1:36]                # one window: 35 time steps x 20 streams; y = the next word
state = lm.init_state(20)                   # zero (h, c) for both layers (Sec. 4.1)

loss, state, cache = lm.forward(x, y, state, training=True)
masks = cache['masks']
print(f"{lm.parameter_count():,} parameters (the paper's 2x200 model: 4,651,600)")
print(f"dropout masks: {len(masks)} = L+1 non-recurrent connections, each of shape {masks[0].shape}; "
      f"fraction kept {np.mean(masks[0] > 0):.2f}")
print(f"loss per word {loss / 35:.3f} nats -> perplexity {np.exp(loss / 35):,.0f} (untrained; uniform = 10,000)")

grads, _ = lm.backward(cache)               # the gradient w.r.t. the carried-in state is dropped: truncation
norm = clip_grad_norm(grads, 5.0)           # one global norm over all parameters, as in the paper
print(f"gradient norm {norm:.2f} -> after clipping {min(norm, 5.0):.2f}")
lm.sgd_update(grads, lr=1.0)
loss2 = lm.forward(x, y, lm.init_state(20), masks=masks, training=True)[0]   # same masks: a fair comparison
print(f"same window after one SGD step: loss per word {loss / 35:.3f} -> {loss2 / 35:.3f}")

# The rule of Sec. 3.2, checked: the state carried to the next window ignores every dropout mask
# above it, so changing the top (layer 2 -> softmax) mask leaves it bit-identical.
other = lm.sample_masks(35, 20)
sA = lm.forward(x, y, lm.init_state(20), masks=masks, training=True)[1]
sB = lm.forward(x, y, lm.init_state(20), masks=[masks[0], masks[1], other[2]], training=True)[1]
print("carried (h, c) unchanged by the output-path mask:",
      all(np.array_equal(a, b) for la, lb in zip(sA, sB) for a, b in zip(la, lb)))

In [ ]:
# The real training loop of train_minimal.py on a tiny budget (well under a minute of CPU).
# Paper recipe: 35-step windows with the state carried across them, batch 20, SGD at lr 1
# (halved per epoch after epoch 2 here), global gradient-norm clipping at 5, 50% dropout.
cfg = dict(hidden=200, layers=2, keep_prob=0.5, init_scale=0.1, bptt=35, batch_size=20, lr=1.0,
           decay_after=2, lr_decay=2.0, max_grad_norm=5.0, epochs=3,
           train_tokens=50_000, valid_tokens=20_000, train_eval_tokens=10_000)
hist = train_lm(cfg, train_ids, valid_ids, vocab_size=len(vocab), seed=0)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(hist['epoch'], hist['train_ppl'], 'b--o', label='train (dropout on, during the epoch)')
plt.plot(hist['epoch'], hist['train_eval_ppl'], 'b-o', label='train (eval mode)')
plt.plot(hist['epoch'], hist['valid_ppl'], 'r-s', label='validation')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('perplexity (log scale)')
plt.title('2x200 LSTM, 50% dropout, first 50k PTB words')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Validation perplexity after {len(hist['epoch'])} epochs: {hist['valid_ppl'][-1]:.0f} "
      f"(about 10,000 untrained); {hist['cpu_seconds']:.0f} s of CPU.")
print("Three epochs on 5% of PTB is far from the paper's 82.7, but the model clearly learns: the")
print("loss falls on held-out text. To see what dropout buys, the next cell compares runs long enough")
print("for an unregularized model to overfit.")

### Dropout vs no dropout: the recorded CPU run

`python train_minimal.py --compare` trains the same 2×200 model twice on the first 100,000 PTB training
words for 10 epochs (lr 1 for 6 epochs, then halved each epoch), once with 50% dropout and once
without; everything else, including the initialization, is identical. On this slice the 4.65M-parameter
model is "large" relative to its data, which is the regime the paper is about. It takes about 10 minutes
of CPU, so the next cell plots the numbers recorded from one run (seed 0); set `RUN_FULL_COMPARISON = True`
to re-run it.

What to look for: without dropout the training perplexity keeps falling while the validation perplexity
turns upward (overfitting); with dropout the two stay close, and the best validation perplexity is lower.

In [ ]:
RUN_FULL_COMPARISON = False     # True: re-run `train_minimal.py --compare` here (~10 min of CPU)

if RUN_FULL_COMPARISON:
    from train_minimal import CPU_COMPARE
    curves = {}
    for keep, name in [(1.0, 'no dropout'), (0.5, '50% dropout')]:
        h = train_lm(dict(CPU_COMPARE, keep_prob=keep), train_ids, valid_ids, test_ids, len(vocab), seed=0)
        curves[name] = {k: h[k] for k in ('train_eval_ppl', 'valid_ppl', 'best_valid_ppl', 'best_epoch',
                                          'test_ppl')}
else:
    # Recorded with `python train_minimal.py --compare` (seed 0), 2026-10-18.
    curves = {
        'no dropout': {
            'train_eval_ppl': [713.6, 475.1, 301.9, 242.6, 176.4, 132.8, 97.1, 82.9, 74.4, 69.8],
            'valid_ppl':      [704.3, 486.6, 361.6, 347.7, 326.5, 323.5, 315.4, 326.7, 331.4, 333.9],
            'best_valid_ppl': 315.4, 'best_epoch': 7, 'test_ppl': 307.8},
        '50% dropout': {
            'train_eval_ppl': [751.4, 501.6, 360.0, 300.1, 234.9, 202.7, 164.7, 149.1, 142.1, 138.8],
            'valid_ppl':      [718.0, 507.8, 402.8, 359.6, 317.1, 302.7, 280.8, 273.1, 269.0, 266.7],
            'best_valid_ppl': 266.7, 'best_epoch': 10, 'test_ppl': 258.8},
    }

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, c) in zip(axes, curves.items()):
    epochs = range(1, len(c['valid_ppl']) + 1)
    ax.plot(epochs, c['train_eval_ppl'], 'b-o', label='train (eval mode)')
    ax.plot(epochs, c['valid_ppl'], 'r-s', label='validation')
    ax.axvline(c['best_epoch'], color='green', ls='--', alpha=0.6, label=f"best validation (epoch {c['best_epoch']})")
    ax.set_title(name)
    ax.set_xlabel('epoch')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel('perplexity (log scale)')
plt.suptitle('2x200 LSTM on the first 100k PTB words, 10 epochs')
plt.tight_layout()
plt.show()

print(f"{'':12s} {'best valid':>10s} {'(epoch)':>8s} {'final valid':>12s} {'final train':>12s} {'test':>7s}")
for name, c in curves.items():
    print(f"{name:12s} {c['best_valid_ppl']:10.1f} {c['best_epoch']:8d} {c['valid_ppl'][-1]:12.1f} "
          f"{c['train_eval_ppl'][-1]:12.1f} {c['test_ppl']:7.1f}")

## Summary

| Technique | Source | When to use | Typical value |
|-----------|--------|-------------|---------------|
| **Dropout** | Zaremba et al. 2014 | Non-recurrent connections only | keep_prob=0.5-0.8 |
| **Layer Norm** | Ba et al. 2016 (our addition) | Deep RNNs, unstable training | Always on |
| **Weight Decay** | Standard (our addition) | All models, overfitting | lambda=0.0001 |
| **Early Stopping** | Standard (our addition) | All models | patience=5 |

The paper's key insight: dropout works for LSTMs if you only apply it to vertical
(non-recurrent) connections. The other three techniques are standard complements.

In [ ]:
print("What we covered:")
print("  - Why standard dropout breaks RNNs (disrupts temporal memory)")
print("  - The paper's fix: dropout only on non-recurrent connections")
print("  - Layer normalization (Ba et al. 2016) for stable training")
print("  - Weight decay (L2) for simpler models")
print("  - Early stopping to prevent overfitting")
print("  - Training the paper's 2-layer LSTM language model on Penn Treebank with BPTT (Part 6)")
print("\nNext: Check out the exercises to practice implementing these.")